# Parcel Address Associator: end-to-end session

Finds **potential duplicate addresses within each cadastre parcel** by working out the address *styles* present (the schematics) and learning how those styles map onto each other. It's built for complex apartment precincts, where one dwelling is often addressed several ways:

| style | example |
|---|---|
| building prefix + level-encoded number, range base | `UNIT A1103, 1-15 MERIDIAN AV` |
| building name | `TOWER A, UNIT 1103, 1-15 MERIDIAN AV` |
| explicit level, unit restarts per level | `UNIT 3, LEVEL 11, 1 MERIDIAN AV` |

Matching is rule-based and statistical only: no LLMs and no libpostal.

**Pipeline** (each stage is a cell below):

```
input DataFrame ─► 1 validate ─► 2 profile ─► 3 run (per parcel) ─► 4 overview
                                                 │
   per parcel: features ─► styles ─► blocks + numbering models ─► block mappings (transform + evidence)
               ─► inferred structures (stacking plans) ─► duplicate pairs ─► duplicate groups + review list
                                                 │
   residential use: base ADDRESS_TYPE + second dataset ─► per-premise reconciliation ─► areas, categories, fixes
                                                 │
   shell addresses: parcel structure (+ parcels, footprints, geocodes) ─► building / site / common property
```

**Input:** one row per address, already linked to a parcel. Columns: `ADDR_ID` (identifier), `SOURCE_ID` (address text identifier), the G-NAF component columns, and `CADASTRE_POLYGON_PID`. Only `ADDR_ID` and `CADASTRE_POLYGON_PID` are required; missing component columns are treated as blank. `LATITUDE`/`LONGITUDE` are optional and only used by shell detection.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))          # project root (this notebook lives in notebooks/)

import pandas as pd
from parcel_address_associator import AssociatorSession, Config, INPUT_COLUMNS, make_synthetic

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 220)
print(len(INPUT_COLUMNS), "input columns:", ", ".join(INPUT_COLUMNS))

27 input columns: ADDR_ID, SOURCE_ID, BUILDING_NAME, LOT_NUMBER_PREFIX, LOT_NUMBER, LOT_NUMBER_SUFFIX, FLAT_TYPE_CODE, FLAT_NUMBER_PREFIX, FLAT_NUMBER, FLAT_NUMBER_SUFFIX, LEVEL_TYPE_CODE, LEVEL_NUMBER_PREFIX, LEVEL_NUMBER, LEVEL_NUMBER_SUFFIX, NUMBER_FIRST_PREFIX, NUMBER_FIRST, NUMBER_FIRST_SUFFIX, NUMBER_LAST_PREFIX, NUMBER_LAST, NUMBER_LAST_SUFFIX, STREET_NAME, STREET_TYPE_CODE, STREET_SUFFIX_CODE, LOCALITY_NAME, STATE_ABBREVIATION, POSTCODE, CADASTRE_POLYGON_PID


## 1 · Input

Swap the synthetic data for your own DataFrame, e.g. `df = spark.table("...").toPandas()` or `df = pd.read_parquet(...)`.

The synthetic set is a fictional town of apartment precincts. Each one reproduces a duplicate pattern from real data, rendered by several "sources" that number the same dwellings differently. `truth` records which physical dwelling each `ADDR_ID` is, and is only used for scoring.

For the residential-use section (11), `df` also carries `ADDRESS_TYPE`, and `use_df` is the second dataset: one row per `ADDR_ID` with a yes/no `RESIDENTIAL_DWELLING` flag. Both contain deliberate errors.

For the shell section (12), `df` also carries `LATITUDE`/`LONGITUDE`, and `geo` holds the cadastre (`CADASTRE_POLYGON_PID`, `WKT`) and building footprints (`BUILDING_ID`, `CADASTRE_POLYGON_PID`, `WKT`) in lon/lat WKT.

In [2]:
df, truth, use_df, geo = make_synthetic(seed=42, with_use=True, with_geometry=True)
# df = <your DataFrame with INPUT_COLUMNS (+ ADDRESS_TYPE, LATITUDE, LONGITUDE)>
# use_df = <your residential dataset: ADDR_ID, RESIDENTIAL_DWELLING>
# geo = {"cadastre": <CADASTRE_POLYGON_PID, WKT>, "buildings": <BUILDING_ID, CADASTRE_POLYGON_PID, WKT>}
print(df.shape, use_df.shape, geo["cadastre"].shape, geo["buildings"].shape)
df.head(8)

(2350, 30) (2350, 2) (65, 2) (90, 3)


,ADDR_ID,SOURCE_ID,BUILDING_NAME,LOT_NUMBER_PREFIX,LOT_NUMBER,LOT_NUMBER_SUFFIX,FLAT_TYPE_CODE,FLAT_NUMBER_PREFIX,FLAT_NUMBER,FLAT_NUMBER_SUFFIX,LEVEL_TYPE_CODE,LEVEL_NUMBER_PREFIX,LEVEL_NUMBER,LEVEL_NUMBER_SUFFIX,NUMBER_FIRST_PREFIX,NUMBER_FIRST,NUMBER_FIRST_SUFFIX,NUMBER_LAST_PREFIX,NUMBER_LAST,NUMBER_LAST_SUFFIX,STREET_NAME,STREET_TYPE_CODE,STREET_SUFFIX_CODE,LOCALITY_NAME,STATE_ABBREVIATION,POSTCODE,CADASTRE_POLYGON_PID,ADDRESS_TYPE,LATITUDE,LONGITUDE
0,A0001957,TXT40025441,,,,,SE,,103,,,,,,,64,,,68,,BANKSIA,STREET,,PORT ALDER,NSW,2989,cpl5a1700013556,COMMERCIAL,-33.099793,150.907574
1,A0000379,TXT40004927,TOWER A,,,,UNIT,,806,,,,,,,1,,,15,,MERIDIAN,AVENUE,,PORT ALDER,NSW,2989,cpl5a1700003dde,SHELL_CHILD,-33.099684,150.901414
2,A0001734,TXT40022542,WEST TOWER,,,,UNIT,,502,,,,,,,50,,,52,,SKYLINE,BOULEVARD,,PORT ALDER,NSW,2989,cpl5a170000f778,SHELL_CHILD,-33.099775,150.905946
3,A0001875,TXT40024375,EAST TOWER,,,,UNIT,,805,,,,,,,50,,,52,,SKYLINE,BOULEVARD,,PORT ALDER,NSW,2989,cpl5a170000f778,SHELL_CHILD,-33.099775,150.905946
4,A0000153,TXT40001989,,,,,UNIT,,1202,,,,,,,2,,,8,,HARBOURVIEW,PARADE,,PORT ALDER,NSW,2989,cpl5a1700001eef,SHELL_CHILD,-33.099793,150.900268
5,A0001931,TXT40025103,,,,,TNHS,,2,,,,,,,88,,,,,ORCHARD,STREET,,PORT ALDER,NSW,2989,cpl5a1700011667,SHELL_CHILD,-33.099486,150.906706
6,A0001499,TXT40019487,,,,,UNIT,A,302,,,,,,,50,,,52,,SKYLINE,BOULEVARD,,PORT ALDER,NSW,2989,cpl5a170000d889,SHELL_CHILD,-33.099775,150.904853
7,A0000855,TXT40011115,,,,,UNIT,B,1201,,,,,,,1,,,15,,MERIDIAN,AVENUE,,PORT ALDER,NSW,2989,cpl5a1700003dde,SHELL_CHILD,-33.099684,150.901414


## 2 · Validate

`validate()` checks the schema and repairs what it safely can. Problems are reported, not fatal:
- numbers stored as floats (`101.0` → `101`);
- identifiers packed into the number field (`G01` → prefix `G` + number `1`);
- names instead of codes (`APARTMENT` → `APT`, `ST` → `STREET`).

It also flags blank or duplicate `ADDR_ID`s, blank parcels and unknown codes.

In [3]:
s = AssociatorSession(df)
s.validate()

validate: 2350 of 2350 rows accepted, 1 issue types (0 error rows)


,CHECK,SEVERITY,ROWS,DETAIL,EXAMPLE_ADDR_IDS
0,rows_accepted,info,2350,2350 of 2350 input rows accepted,


What the checks look like on messy input (a few rows deliberately broken):

In [4]:
messy = df.head(6).astype(object)
messy["LEVEL_NUMBER"] = pd.to_numeric(messy["LEVEL_NUMBER"], errors="coerce")  # float column with NaN
messy.loc[0, "FLAT_NUMBER"] = 101.0                      # number stored as float
messy.loc[1, ["FLAT_NUMBER_PREFIX", "FLAT_NUMBER"]] = ["", "G01"]   # packed identifier
messy.loc[2, "FLAT_TYPE_CODE"] = "APARTMENT"             # name instead of code
messy.loc[3, "STREET_TYPE_CODE"] = "ST"                  # abbreviation instead of code
messy.loc[4, "ADDR_ID"] = messy.loc[5, "ADDR_ID"]        # duplicate id
messy = messy.drop(columns=["SOURCE_ID"])                # missing optional column
AssociatorSession(messy, verbose=False).validate()

,CHECK,SEVERITY,ROWS,DETAIL,EXAMPLE_ADDR_IDS
0,missing_column,warning,6,SOURCE_ID not supplied; treated as blank,
1,duplicate_addr_id,error,1,repeated ADDR_ID; first occurrence kept,A0001931
2,code_normalised,info,1,FLAT_TYPE_CODE names/abbreviations mapped to G-NAF codes (e.g. APARTMENT->APT),
3,code_normalised,info,1,STREET_TYPE_CODE names/abbreviations mapped to G-NAF codes (e.g. ST->STREET),
4,identifier_split,info,1,FLAT_NUMBER held prefix/suffix letters (e.g. G01); split into FLAT_NUMBER_PR...,A0000379
5,rows_accepted,info,5,5 of 6 input rows accepted,


## 3 · Profile

What each parcel holds before any matching. Parcels with fewer than `Config.min_addresses` addresses (default 2) are skipped, because there is nothing to deduplicate.

In [5]:
s.profile(top=10)

profile: 65 parcels, 32 with >= 2 addresses will be analysed


,CADASTRE_POLYGON_PID,N_ADDRESSES,N_SUB_ADDRESSES,N_WITH_LEVEL,N_WITH_BUILDING_NAME,N_RANGE_ADDRESSES,N_DISTINCT_BASES,FLAT_TYPES,WILL_ANALYSE
0,cpl5a1700003dde,790,783,0,261,593,4,"UNIT:773, SHOP:10",True
1,cpl5a170000f778,212,207,0,89,90,3,UNIT:207,True
2,cpl5a170000d889,210,207,0,89,210,1,UNIT:207,True
3,cpl5a1700001eef,201,198,66,1,134,2,"UNIT:178, APT:13, SHOP:7",True
4,cpl5a1700013556,168,166,1,0,105,2,"UNIT:88, APT:51, SE:10, OFFC:10, SHOP:6",True
5,cpl5a170000b99a,159,157,65,0,49,2,"UNIT:122, APT:35",True
6,cpl5a1700007bbc,118,117,0,0,0,1,UNIT:117,True
7,cpl5a1700005ccd,117,115,24,0,0,2,UNIT:115,True
8,cpl5a1700015445,97,95,26,1,0,1,"UNIT:73, CARS:12, STOR:6, SUBS:1, CARP:1",True
9,cpl5a1700009aab,91,90,36,0,0,1,UNIT:90,True


## 4 · Run

Every parcel is analysed independently: features → styles → blocks → block mappings → structures → pairs → groups. `s.log` records each stage.

In [6]:
s.run()
s.log

run: 32 parcels analysed in 0.31s, 714 duplicate groups


,STAGE,SECONDS,DETAIL
0,validate,0.184,"rows_in=2350, rows_accepted=2350, issues=1, error_rows=0"
1,profile,0.019,"parcels=65, parcels_to_analyse=32"
2,run,0.311,"parcels=32, groups=714, review_pairs=233"


## 5 · Overview

In [7]:
s.overview()

,METRIC,VALUE
0,input rows,2350
1,rows accepted,2350
2,parcels analysed,32
3,addresses analysed,2317
4,duplicate groups,714
5,addresses in duplicate groups,1644
6,redundant addresses (group size - 1),930
7,review candidate pairs,233


In [8]:
s.parcel_summary().head(10)

,CADASTRE_POLYGON_PID,N_ADDRESSES,N_SUB_ADDRESSES,N_PARENTS,N_STYLES,N_BLOCKS,N_STRUCTURES,N_DUPLICATE_GROUPS,N_ADDRESSES_IN_GROUPS,N_REDUNDANT_ADDRESSES,N_REVIEW_PAIRS,REJECTED_CONFLICTS
0,cpl5a1700003dde,790,783,7,5,12,3,305,760,455,0,0
1,cpl5a170000d889,210,207,3,2,4,2,87,174,87,0,0
2,cpl5a1700001eef,201,198,3,5,5,1,75,159,84,0,0
3,cpl5a1700013556,168,166,2,6,6,1,63,126,63,0,0
4,cpl5a170000b99a,159,157,2,4,4,1,50,149,99,0,0
5,cpl5a1700005ccd,117,115,2,2,3,1,44,96,52,0,0
6,cpl5a1700009aab,91,90,1,2,2,1,36,72,36,0,0
7,cpl5a1700015445,97,95,2,8,8,6,25,50,25,0,0
8,cpl5a1700011667,45,43,2,3,4,2,11,22,11,5,0
9,cpl5a1700036224,12,11,1,2,2,1,5,10,5,0,0


## 6 · One parcel, stage by stage

Taking the parcel with the most duplicate groups. Vocabulary used below:

- **Style:** a formatting scheme, e.g. `UNIT {B}{L}{UU}, {N}-{N} {STREET}` (building prefix + level-encoded unit on a range base).
- **Block:** the addresses of one style on one context (base address + building reference). It is what that scheme says is in one building.
- **Numbering model:** how a block's numbers encode position: `encoded` (1103 = level 11 unit 3), `explicit+per-level` (`UNIT 3, LEVEL 11`), `explicit+sequential`, `prefix-level` (G01), `plain`.
- **Mapping:** for each pair of blocks, the transform that explains the most records (same identifier; level + unit-on-level, optionally with a level offset; position within level; position through the building). It is scored in log-odds from coverage, layout extent, building references and base-address relationship.
- **Structure:** linked blocks clustered into an inferred building, drawn as a stacking plan.

In [9]:
pid = s.interesting_parcels(1)[0]
print(s.schematic(pid))

PARCEL cpl5a1700003dde  790 addresses (783 sub-addresses, 7 parents) | 305 duplicate groups | 0 review
  STYLES
    S1   UNIT {B}{L}{UU}, {N}-{N} {STREET}                         324  UNIT|BPFX|encoded|RANGE
         S1.1    1-15 MERIDIAN AV | prefix A            n=180  unit number encodes level (1103 = level 11, unit 3); levels L1-L18, ~10 per level
         S1.2    1-15 MERIDIAN AV | prefix B            n=96   unit number encodes level (1103 = level 11, unit 3); levels L1-L12, ~8 per level
         S1.3    1-15 MERIDIAN AV | prefix C            n=48   unit number encodes level (1103 = level 11, unit 3); levels L1-L8, ~6 per level
    S2   TOWER {B}, UNIT {L}{UU}, {N}-{N} {STREET}                 258  UNIT|encoded|BN|RANGE
         S2.1    1-15 MERIDIAN AV | TOWER A             n=144  unit number encodes level (1103 = level 11, unit 3); levels L1-L18, ~8 per level
         S2.2    1-15 MERIDIAN AV | TOWER B             n=79   unit number encodes level (1103 = level 11, unit 3); levels

### 6a · Styles on the parcel

In [10]:
s.styles(pid)

,CADASTRE_POLYGON_PID,STYLE_ID,TEMPLATE,SIGNATURE,N_ADDRESSES,N_BLOCKS,BLOCKS,EXAMPLES
0,cpl5a1700003dde,S1,"UNIT {B}{L}{UU}, {N}-{N} {STREET}",UNIT|BPFX|encoded|RANGE,324,3,S1.1: 1-15 MERIDIAN AV | prefix A [encoded] n=180; S1.2: 1-15 MERIDIAN AV | ...,"UNIT B1201, 1-15 MERIDIAN AV | UNIT B1203, 1-15 MERIDIAN AV | UNIT A1105, 1-..."
1,cpl5a1700003dde,S2,"TOWER {B}, UNIT {L}{UU}, {N}-{N} {STREET}",UNIT|encoded|BN|RANGE,258,3,S2.1: 1-15 MERIDIAN AV | TOWER A [encoded] n=144; S2.2: 1-15 MERIDIAN AV | T...,"TOWER A, UNIT 806, 1-15 MERIDIAN AV | TOWER A, UNIT 1505, 1-15 MERIDIAN AV |..."
2,cpl5a1700003dde,S3,"UNIT {L}{UU}, {N} {STREET}",UNIT|encoded|SINGLE,191,3,S3.1: 1 MERIDIAN AV [encoded] n=102; S3.2: 15 MERIDIAN AV [encoded] n=28; S3...,"UNIT 1505, 1 MERIDIAN AV | UNIT 804, 1 MERIDIAN AV | UNIT 1502, 1 MERIDIAN AV"
3,cpl5a1700003dde,S4,"SHOP {N}, {N}-{N} {STREET}",SHOP|plain|RANGE,7,1,S4.1: 1-15 MERIDIAN AV [plain] n=7,"SHOP 7, 1-15 MERIDIAN AV | SHOP 4, 1-15 MERIDIAN AV | SHOP 1, 1-15 MERIDIAN AV"
4,cpl5a1700003dde,S5,"SHOP {N}, {N} {STREET}",SHOP|plain|SINGLE,3,2,S5.1: 1 MERIDIAN AV [plain] n=2; S5.2: 7 MERIDIAN AV [plain] n=1,"SHOP 2, 1 MERIDIAN AV | SHOP 3, 1 MERIDIAN AV | SHOP 6, 7 MERIDIAN AV"


### 6b · Blocks and numbering models

In [11]:
s.blocks(pid)

,CADASTRE_POLYGON_PID,BLOCK_ID,STYLE_ID,CONTEXT,BUILDING_REF,FLAT_TYPE,NUMBERING_MODEL,MODEL_NOTE,N_ADDRESSES,LEVELS,UNITS_PER_LEVEL,STRUCTURE_ID,SPANS_BUILDINGS,KEY_FORMS
0,cpl5a1700003dde,S1.1,S1,1-15 MERIDIAN AV | prefix A,A,UNIT,encoded,"unit number encodes level (1103 = level 11, unit 3); levels L1-L18, ~10 per ...",180,L1-L18,10.0,B1,False,"raw,lu,lr,seq"
1,cpl5a1700003dde,S1.2,S1,1-15 MERIDIAN AV | prefix B,B,UNIT,encoded,"unit number encodes level (1103 = level 11, unit 3); levels L1-L12, ~8 per l...",96,L1-L12,8.0,B2,False,"raw,lu,lr,seq"
2,cpl5a1700003dde,S1.3,S1,1-15 MERIDIAN AV | prefix C,C,UNIT,encoded,"unit number encodes level (1103 = level 11, unit 3); levels L1-L8, ~6 per level",48,L1-L8,6.0,B3,False,"raw,lu,lr,seq"
3,cpl5a1700003dde,S2.1,S2,1-15 MERIDIAN AV | TOWER A,A,UNIT,encoded,"unit number encodes level (1103 = level 11, unit 3); levels L1-L18, ~8 per l...",144,L1-L18,8.0,B1,False,"raw,lu,lr,seq"
4,cpl5a1700003dde,S2.2,S2,1-15 MERIDIAN AV | TOWER B,B,UNIT,encoded,"unit number encodes level (1103 = level 11, unit 3); levels L1-L12, ~7 per l...",79,L1-L12,7.0,B2,False,"raw,lu,lr,seq"
5,cpl5a1700003dde,S2.3,S2,1-15 MERIDIAN AV | TOWER C,C,UNIT,encoded,"unit number encodes level (1103 = level 11, unit 3); levels L1-L8, ~5 per level",35,L1-L8,5.0,B3,False,"raw,lu,lr,seq"
6,cpl5a1700003dde,S3.1,S3,1 MERIDIAN AV,,UNIT,encoded,"unit number encodes level (1103 = level 11, unit 3); levels L1-L18, ~6 per l...",102,L1-L18,6.0,B1,False,"raw,lu,lr,seq"
7,cpl5a1700003dde,S3.2,S3,15 MERIDIAN AV,,UNIT,encoded,"unit number encodes level (1103 = level 11, unit 3); levels L1-L8, ~3.5 per ...",28,L1-L8,3.5,B3,False,"raw,lu,lr,seq"
8,cpl5a1700003dde,S3.3,S3,7 MERIDIAN AV,,UNIT,encoded,"unit number encodes level (1103 = level 11, unit 3); levels L1-L12, ~5.5 per...",61,L1-L12,5.5,B2,False,"raw,lu,lr,seq"
9,cpl5a1700003dde,S4.1,S4,1-15 MERIDIAN AV,,SHOP,plain,"plain sequence, no level information",7,,NaN,,True,"raw,seq,num"


### 6c · How the blocks map to each other

`STATUS`:
- `KEPT`: used for duplicates.
- `REJECTED`: the evidence says these are different buildings or premises.
- `AMBIGUOUS`: fits two buildings equally well.
- `REVIEW`: an order-based mapping that components alone can't confirm.

`EVIDENCE` lists each log-odds contribution.

In [12]:
s.mappings(pid, include_rejected=True)

,CADASTRE_POLYGON_PID,LINK_ID,BLOCK_A,CONTEXT_A,BLOCK_B,CONTEXT_B,TRANSFORM,MATCHED_KEYS,KEYS_A,KEYS_B,QUALITY,P,BAND,STATUS,EVIDENCE
0,cpl5a1700003dde,S1.1~S2.1,S1.1,1-15 MERIDIAN AV | prefix A,S2.1,1-15 MERIDIAN AV | TOWER A,same unit identifier,144,180,144,1.000,0.9980,HIGH,KEPT,same unit identifier explains 144 of 180/144 keys (+3.2); building ref A = A...
1,cpl5a1700003dde,S1.2~S2.2,S1.2,1-15 MERIDIAN AV | prefix B,S2.2,1-15 MERIDIAN AV | TOWER B,same unit identifier,79,96,79,1.000,0.9980,HIGH,KEPT,same unit identifier explains 79 of 96/79 keys (+3.2); building ref B = B (+...
2,cpl5a1700003dde,S1.3~S2.3,S1.3,1-15 MERIDIAN AV | prefix C,S2.3,1-15 MERIDIAN AV | TOWER C,same unit identifier,35,48,35,1.000,0.9980,HIGH,KEPT,same unit identifier explains 35 of 48/35 keys (+3.2); building ref C = C (+...
3,cpl5a1700003dde,S1.1~S3.1,S1.1,1-15 MERIDIAN AV | prefix A,S3.1,1 MERIDIAN AV,same unit identifier,102,180,102,1.000,0.9820,HIGH,KEPT,same unit identifier explains 102 of 180/102 keys (+3.2); range contains num...
4,cpl5a1700003dde,S1.2~S3.3,S1.2,1-15 MERIDIAN AV | prefix B,S3.3,7 MERIDIAN AV,same unit identifier,61,96,61,1.000,0.9820,HIGH,KEPT,same unit identifier explains 61 of 96/61 keys (+3.2); range contains number...
5,cpl5a1700003dde,S1.3~S3.2,S1.3,1-15 MERIDIAN AV | prefix C,S3.2,15 MERIDIAN AV,same unit identifier,28,48,28,1.000,0.9820,HIGH,KEPT,same unit identifier explains 28 of 48/28 keys (+3.2); range contains number...
6,cpl5a1700003dde,S2.1~S3.1,S2.1,1-15 MERIDIAN AV | TOWER A,S3.1,1 MERIDIAN AV,same unit identifier,83,144,102,0.888,0.9571,HIGH,KEPT,same unit identifier explains 83 of 144/102 keys (+2.3); range contains numb...
7,cpl5a1700003dde,S2.2~S3.3,S2.2,1-15 MERIDIAN AV | TOWER B,S3.3,7 MERIDIAN AV,same unit identifier,47,79,61,0.862,0.9478,HIGH,KEPT,same unit identifier explains 47 of 79/61 keys (+2.1); range contains number...
8,cpl5a1700003dde,S2.3~S3.2,S2.3,1-15 MERIDIAN AV | TOWER C,S3.2,15 MERIDIAN AV,same unit identifier,20,35,28,0.829,0.9327,HIGH,KEPT,same unit identifier explains 20 of 35/28 keys (+1.8); range contains number...
9,cpl5a1700003dde,S4.1~S5.1,S4.1,1-15 MERIDIAN AV,S5.1,1 MERIDIAN AV,same unit identifier,2,7,2,0.714,0.7711,MEDIUM,KEPT,same unit identifier explains 2 of 7/2 keys (+0.9); only 2 matched (-0.5); r...


### 6d · Inferred structures

In [13]:
s.structures(pid)

,CADASTRE_POLYGON_PID,STRUCTURE_ID,LABEL,BLOCKS,CLASSES,LEVELS,N_SLOTS,N_SLOTS_WITH_DUPLICATES,N_UNPLACED
0,cpl5a1700003dde,B1,A | 1 MERIDIAN AV,"S1.1,S2.1,S3.1,S5.1","COMMERCIAL,RESIDENTIAL",L1-L18,180,163,2
1,cpl5a1700003dde,B2,B | 7 MERIDIAN AV,"S1.2,S2.2,S3.3,S5.2","COMMERCIAL,RESIDENTIAL",L1-L12,96,93,1
2,cpl5a1700003dde,B3,C | 15 MERIDIAN AV,"S1.3,S2.3,S3.2",RESIDENTIAL,L1-L8,48,43,0


### 6e · The full schematic, saved as HTML

`save_html()` writes a standalone page and returns its path; nothing is opened or displayed. Open the file in any browser. The page shows styles, mappings, a stacking plan per inferred building, parents and groups. In each stacking-plan cell, the coloured stripes show which styles address that slot (the stripe position is fixed per style) and the number is how many addresses fall in it. Hover a cell to list them.

(For an inline view in the notebook instead, use `s.show_parcel(pid)`.)

In [14]:
_ = s.save_html("../outputs/parcel_%s.html" % pid, parcels=[pid])

saved ..\outputs\parcel_cpl5a1700003dde.html (1 parcels in detail, 221 KB)


A parcel the tool *can't* resolve: two identical towers where one source uses street numbers and the other uses EAST/WEST TOWER names. Nothing in the components says which is which, so the pairs go to the review list instead of being guessed.

In [15]:
amb = s.parcel_summary().sort_values("N_REVIEW_PAIRS", ascending=False).CADASTRE_POLYGON_PID.iloc[0]
_ = s.save_html("../outputs/parcel_%s.html" % amb, parcels=[amb])

saved ..\outputs\parcel_cpl5a170000f778.html (1 parcels in detail, 94 KB)


## 7 · Results

- `duplicate_groups()`: one row per group, with a suggested record to keep. The suggestion comes from the parcel's dominant style, then the most complete record.
- `duplicate_pairs()`: the evidence behind each grouped pair.
- `review_candidates()`: plausible pairs that components can't confirm. These are never grouped.
- `addresses()`: every analysed input row annotated with style, block, structure, slot and group.

In [16]:
groups = s.duplicate_groups()
print(groups.BAND.value_counts().to_dict(), "|", groups.RELATION.value_counts().to_dict())
groups.head(10)

{'HIGH': 624, 'MEDIUM': 61, 'LOW': 29} | {'SAME_DWELLING': 704, 'SAME_BUILDING': 7, 'SAME_PROPERTY': 2, 'EXACT_DUPLICATE': 1}


,CADASTRE_POLYGON_PID,GROUP_ID,SIZE,RELATION,BAND,P_MIN,P_MAX,PREFERRED_ADDR_ID,PREFERRED_LABEL,MEMBER_ADDR_IDS,MEMBER_SOURCE_IDS,MEMBER_LABELS,MEMBER_STYLES
0,cpl5a1700013556,G1,2,SAME_BUILDING,MEDIUM,0.8455,0.8455,A0001945,64 BANKSIA ST,A0001944;A0001945,TXT40025272;TXT40025285,64-68 BANKSIA ST | 64 BANKSIA ST,parent;parent
1,cpl5a1700013556,G2,2,SAME_DWELLING,LOW,0.5083,0.5083,A0001947,"SHOP 2, 64-68 BANKSIA ST",A0001947;A0001948,TXT40025311;TXT40025324,"SHOP 2, 64-68 BANKSIA ST | TENANCY 2, GROUND, 64 BANKSIA ST",S5;S6
2,cpl5a1700013556,G3,2,SAME_DWELLING,HIGH,0.9759,0.9759,A0001954,"OFFICE 101, 64 BANKSIA ST",A0001953;A0001954,TXT40025389;TXT40025402,"SUITE 101, 64-68 BANKSIA ST | OFFICE 101, 64 BANKSIA ST",S4;S3
3,cpl5a1700013556,G4,2,SAME_DWELLING,HIGH,0.9759,0.9759,A0001956,"OFFICE 102, 64 BANKSIA ST",A0001955;A0001956,TXT40025415;TXT40025428,"SUITE 102, 64-68 BANKSIA ST | OFFICE 102, 64 BANKSIA ST",S4;S3
4,cpl5a1700013556,G5,2,SAME_DWELLING,HIGH,0.9759,0.9759,A0001958,"OFFICE 103, 64 BANKSIA ST",A0001957;A0001958,TXT40025441;TXT40025454,"SUITE 103, 64-68 BANKSIA ST | OFFICE 103, 64 BANKSIA ST",S4;S3
5,cpl5a1700013556,G6,2,SAME_DWELLING,HIGH,0.9759,0.9759,A0001960,"OFFICE 104, 64 BANKSIA ST",A0001959;A0001960,TXT40025467;TXT40025480,"SUITE 104, 64-68 BANKSIA ST | OFFICE 104, 64 BANKSIA ST",S4;S3
6,cpl5a1700013556,G7,2,SAME_DWELLING,HIGH,0.9759,0.9759,A0001962,"OFFICE 105, 64 BANKSIA ST",A0001961;A0001962,TXT40025493;TXT40025506,"SUITE 105, 64-68 BANKSIA ST | OFFICE 105, 64 BANKSIA ST",S4;S3
7,cpl5a1700013556,G8,2,SAME_DWELLING,HIGH,0.9759,0.9759,A0001964,"OFFICE 106, 64 BANKSIA ST",A0001963;A0001964,TXT40025519;TXT40025532,"SUITE 106, 64-68 BANKSIA ST | OFFICE 106, 64 BANKSIA ST",S4;S3
8,cpl5a1700013556,G9,2,SAME_DWELLING,HIGH,0.9759,0.9759,A0001966,"OFFICE 107, 64 BANKSIA ST",A0001965;A0001966,TXT40025545;TXT40025558,"SUITE 107, 64-68 BANKSIA ST | OFFICE 107, 64 BANKSIA ST",S4;S3
9,cpl5a1700013556,G10,2,SAME_DWELLING,HIGH,0.9759,0.9759,A0001968,"OFFICE 108, 64 BANKSIA ST",A0001967;A0001968,TXT40025571;TXT40025584,"SUITE 108, 64-68 BANKSIA ST | OFFICE 108, 64 BANKSIA ST",S4;S3


In [17]:
s.duplicate_pairs().head(10)

,CADASTRE_POLYGON_PID,GROUP_ID,ADDR_ID_A,SOURCE_ID_A,LABEL_A,STYLE_A,BLOCK_A,ADDR_ID_B,SOURCE_ID_B,LABEL_B,STYLE_B,BLOCK_B,RELATION,METHOD,P,BAND,EVIDENCE
0,cpl5a1700013556,G3,A0001954,TXT40025402,"OFFICE 101, 64 BANKSIA ST",S3,S3.1,A0001953,TXT40025389,"SUITE 101, 64-68 BANKSIA ST",S4,S4.1,SAME_DWELLING,same unit identifier,0.9759,HIGH,same unit identifier explains 10 of 10/10 keys (+3.2); flat types OFFC/SE (-...
1,cpl5a1700013556,G4,A0001956,TXT40025428,"OFFICE 102, 64 BANKSIA ST",S3,S3.1,A0001955,TXT40025415,"SUITE 102, 64-68 BANKSIA ST",S4,S4.1,SAME_DWELLING,same unit identifier,0.9759,HIGH,same unit identifier explains 10 of 10/10 keys (+3.2); flat types OFFC/SE (-...
2,cpl5a1700013556,G5,A0001958,TXT40025454,"OFFICE 103, 64 BANKSIA ST",S3,S3.1,A0001957,TXT40025441,"SUITE 103, 64-68 BANKSIA ST",S4,S4.1,SAME_DWELLING,same unit identifier,0.9759,HIGH,same unit identifier explains 10 of 10/10 keys (+3.2); flat types OFFC/SE (-...
3,cpl5a1700013556,G6,A0001960,TXT40025480,"OFFICE 104, 64 BANKSIA ST",S3,S3.1,A0001959,TXT40025467,"SUITE 104, 64-68 BANKSIA ST",S4,S4.1,SAME_DWELLING,same unit identifier,0.9759,HIGH,same unit identifier explains 10 of 10/10 keys (+3.2); flat types OFFC/SE (-...
4,cpl5a1700013556,G7,A0001962,TXT40025506,"OFFICE 105, 64 BANKSIA ST",S3,S3.1,A0001961,TXT40025493,"SUITE 105, 64-68 BANKSIA ST",S4,S4.1,SAME_DWELLING,same unit identifier,0.9759,HIGH,same unit identifier explains 10 of 10/10 keys (+3.2); flat types OFFC/SE (-...
5,cpl5a1700013556,G8,A0001964,TXT40025532,"OFFICE 106, 64 BANKSIA ST",S3,S3.1,A0001963,TXT40025519,"SUITE 106, 64-68 BANKSIA ST",S4,S4.1,SAME_DWELLING,same unit identifier,0.9759,HIGH,same unit identifier explains 10 of 10/10 keys (+3.2); flat types OFFC/SE (-...
6,cpl5a1700013556,G9,A0001966,TXT40025558,"OFFICE 107, 64 BANKSIA ST",S3,S3.1,A0001965,TXT40025545,"SUITE 107, 64-68 BANKSIA ST",S4,S4.1,SAME_DWELLING,same unit identifier,0.9759,HIGH,same unit identifier explains 10 of 10/10 keys (+3.2); flat types OFFC/SE (-...
7,cpl5a1700013556,G10,A0001968,TXT40025584,"OFFICE 108, 64 BANKSIA ST",S3,S3.1,A0001967,TXT40025571,"SUITE 108, 64-68 BANKSIA ST",S4,S4.1,SAME_DWELLING,same unit identifier,0.9759,HIGH,same unit identifier explains 10 of 10/10 keys (+3.2); flat types OFFC/SE (-...
8,cpl5a1700013556,G11,A0001970,TXT40025610,"OFFICE 109, 64 BANKSIA ST",S3,S3.1,A0001969,TXT40025597,"SUITE 109, 64-68 BANKSIA ST",S4,S4.1,SAME_DWELLING,same unit identifier,0.9759,HIGH,same unit identifier explains 10 of 10/10 keys (+3.2); flat types OFFC/SE (-...
9,cpl5a1700013556,G12,A0001972,TXT40025636,"OFFICE 110, 64 BANKSIA ST",S3,S3.1,A0001971,TXT40025623,"SUITE 110, 64-68 BANKSIA ST",S4,S4.1,SAME_DWELLING,same unit identifier,0.9759,HIGH,same unit identifier explains 10 of 10/10 keys (+3.2); flat types OFFC/SE (-...


In [18]:
rc = s.review_candidates()
print(rc.RELATION.value_counts().to_dict())
rc.head(6)

{'AMBIGUOUS_BUILDING': 174, 'REVIEW_RENUMBERING': 59}


,CADASTRE_POLYGON_PID,GROUP_ID,ADDR_ID_A,SOURCE_ID_A,LABEL_A,STYLE_A,BLOCK_A,ADDR_ID_B,SOURCE_ID_B,LABEL_B,STYLE_B,BLOCK_B,RELATION,METHOD,P,BAND,EVIDENCE
0,cpl5a170000f778,,A0001690,TXT40021970,"UNIT 101, 50 SKYLINE BVD",S1,S1.1,A0001798,TXT40023374,"EAST TOWER, UNIT 101, 50-52 SKYLINE BVD",S2,S2.1,AMBIGUOUS_BUILDING,same unit identifier,0.6874,REVIEW,same unit identifier explains 42 of 60/42 keys (+3.2); range contains number...
1,cpl5a170000f778,,A0001690,TXT40021970,"UNIT 101, 50 SKYLINE BVD",S1,S1.1,A0001691,TXT40021983,"WEST TOWER, UNIT 101, 50-52 SKYLINE BVD",S2,S2.2,AMBIGUOUS_BUILDING,same unit identifier,0.6874,REVIEW,same unit identifier explains 45 of 60/45 keys (+3.2); range contains number...
2,cpl5a170000f778,,A0001692,TXT40021996,"UNIT 102, 50 SKYLINE BVD",S1,S1.1,A0001800,TXT40023400,"EAST TOWER, UNIT 102, 50-52 SKYLINE BVD",S2,S2.1,AMBIGUOUS_BUILDING,same unit identifier,0.6874,REVIEW,same unit identifier explains 42 of 60/42 keys (+3.2); range contains number...
3,cpl5a170000f778,,A0001692,TXT40021996,"UNIT 102, 50 SKYLINE BVD",S1,S1.1,A0001693,TXT40022009,"WEST TOWER, UNIT 102, 50-52 SKYLINE BVD",S2,S2.2,AMBIGUOUS_BUILDING,same unit identifier,0.6874,REVIEW,same unit identifier explains 45 of 60/45 keys (+3.2); range contains number...
4,cpl5a170000f778,,A0001694,TXT40022022,"UNIT 103, 50 SKYLINE BVD",S1,S1.1,A0001802,TXT40023426,"EAST TOWER, UNIT 103, 50-52 SKYLINE BVD",S2,S2.1,AMBIGUOUS_BUILDING,same unit identifier,0.6874,REVIEW,same unit identifier explains 42 of 60/42 keys (+3.2); range contains number...
5,cpl5a170000f778,,A0001695,TXT40022035,"UNIT 104, 50 SKYLINE BVD",S1,S1.1,A0001696,TXT40022048,"WEST TOWER, UNIT 104, 50-52 SKYLINE BVD",S2,S2.2,AMBIGUOUS_BUILDING,same unit identifier,0.6874,REVIEW,same unit identifier explains 45 of 60/45 keys (+3.2); range contains number...


In [19]:
annotated = s.addresses()
annotated[annotated.GROUP_ID != ""].head(10)

,ADDR_ID,SOURCE_ID,CADASTRE_POLYGON_PID,LABEL,KIND,STYLE_ID,STYLE_TEMPLATE,BLOCK_ID,NUMBERING_MODEL,STRUCTURE_ID,LEVEL,UNIT,PARENT_ROLE,GROUP_ID,IS_PREFERRED,DUPLICATE_OF,GROUP_P_MIN,GROUP_BAND
0,A0001957,TXT40025441,cpl5a1700013556,"SUITE 103, 64-68 BANKSIA ST",SUB,S4,"SUITE {L}{UU}, {N}-{N} {STREET}",S4.1,encoded,B1,L1,3,,G5,False,A0001958,0.9759,HIGH
1,A0001944,TXT40025272,cpl5a1700013556,64-68 BANKSIA ST,PARENT,,,,,B1,,,BUILDING,G1,False,A0001945,0.8455,MEDIUM
2,A0002002,TXT40026026,cpl5a1700013556,"UNIT 402, 64-68 BANKSIA ST",SUB,S1,"UNIT {L}{UU}, {N}-{N} {STREET}",S1.1,encoded,B1,L4,2,,G25,True,,0.9759,HIGH
3,A0002027,TXT40026351,cpl5a1700013556,"APARTMENT 601, 64 BANKSIA ST",SUB,S2,"APARTMENT {L}{UU}, {N} {STREET}",S2.1,encoded,B1,L6,1,,G34,False,A0002026,0.9759,HIGH
4,A0002103,TXT40027339,cpl5a1700013556,"APARTMENT 1202, 64 BANKSIA ST",SUB,S2,"APARTMENT {L}{UU}, {N} {STREET}",S2.1,encoded,B1,L12,2,,G61,False,A0002102,0.9759,HIGH
5,A0002067,TXT40026871,cpl5a1700013556,"APARTMENT 901, 64 BANKSIA ST",SUB,S2,"APARTMENT {L}{UU}, {N} {STREET}",S2.1,encoded,B1,L9,1,,G50,False,A0002066,0.9759,HIGH
6,A0001966,TXT40025558,cpl5a1700013556,"OFFICE 107, 64 BANKSIA ST",SUB,S3,"OFFICE {L}{UU}, {N} {STREET}",S3.1,encoded,B1,L1,7,,G9,True,,0.9759,HIGH
7,A0002064,TXT40026832,cpl5a1700013556,"APARTMENT 807, 64 BANKSIA ST",SUB,S2,"APARTMENT {L}{UU}, {N} {STREET}",S2.1,encoded,B1,L8,7,,G49,False,A0002063,0.9759,HIGH
8,A0002010,TXT40026130,cpl5a1700013556,"UNIT 407, 64-68 BANKSIA ST",SUB,S1,"UNIT {L}{UU}, {N}-{N} {STREET}",S1.1,encoded,B1,L4,7,,G28,True,,0.9759,HIGH
9,A0001975,TXT40025675,cpl5a1700013556,"APARTMENT 202, 64 BANKSIA ST",SUB,S2,"APARTMENT {L}{UU}, {N} {STREET}",S2.1,encoded,B1,L2,2,,G13,False,A0001974,0.9759,HIGH


## 8 · Explain one address

In [20]:
aid = annotated[(annotated.KIND == "SUB") & (annotated.GROUP_ID != "")].ADDR_ID.iloc[0]
e = s.explain(aid)
e["PAIRS"]

ADDR_ID          A0001957
SOURCE_ID        TXT40025441
PARCEL           cpl5a1700013556
LABEL            SUITE 103, 64-68 BANKSIA ST
KIND             sub-address
STYLE            S4  SUITE {L}{UU}, {N}-{N} {STREET}
BLOCK            S4.1  64-68 BANKSIA ST
NUMBERING_MODEL  unit number encodes level (1103 = level 11, unit 3); level L1, ~10 per level
KEYS             {'raw': '103', 'lu': (1.0, '3'), 'lr': (1.0, 3), 'seq': 3}
STRUCTURE        B1
PARENT_ROLE      
GROUP            G5 (HIGH, preferred A0001958)


,OTHER,OTHER_LABEL,RELATION,P,METHOD
0,A0001958,"OFFICE 103, 64 BANKSIA ST",SAME_DWELLING,0.976,same unit identifier


In [21]:
e["BLOCK_LINKS"]

,LINK,TRANSFORM,P,STATUS,EVIDENCE
0,S3.1~S4.1,same unit identifier,0.976,KEPT,same unit identifier explains 10 of 10/10 keys (+3.2); flat types OFFC/SE (-...
1,S4.1~S5.1,unit number = position in the other scheme (e.g. strata lot order),0.029,REJECTED,unit number = position in the other scheme (e.g. strata lot order) explains ...


## 9 · Evaluate (synthetic data only)

Scored on the pairs implied by the duplicate groups. Two scenarios are *meant* to have zero grouped recall:
- `strata_lot_numbering`: needs strata-lot IDs to confirm;
- `twin_towers_unresolved`: nothing separates the towers.

Their true pairs should all appear in the review list.

In [22]:
ev = s.evaluate(truth)
ev["by_scenario"]

precision 1.0000  recall 0.8904  (TP 1162, FP 0, FN 143); review candidates 233 (141 true); recall if reviewed 0.9985


,SCENARIO,PRECISION,RECALL,F1,TP,FP,FN,REVIEW_PAIRS,REVIEW_TRUE
0,background,1.0,1.0,1.0,18,0,0,0,0
1,common_property,1.0,1.0,1.0,25,0,0,0,0
2,corner_frontage,1.0,1.0,1.0,60,0,0,0,0
3,level_convention,1.0,1.0,1.0,36,0,0,0,0
4,mixed_use,1.0,1.0,1.0,63,0,0,0,0
5,multi_tower,1.0,1.0,1.0,605,0,0,0,0
6,single_tower,1.0,1.0,1.0,94,0,0,0,0
7,staggered_coverage,1.0,1.0,1.0,163,0,0,0,0
8,strata_lot_numbering,1.0,0.0,0.0,0,0,54,54,54
9,townhouse_mix,1.0,1.0,1.0,11,0,0,5,0


## 10 · Tune and re-run

Every threshold and evidence weight lives in `Config`. `config.with_(...)` returns a modified copy. Re-running a subset of parcels replaces just those results.

In [23]:
print({k: v for k, v in s.config.to_dict().items() if k in ("link_threshold", "band_low", "band_medium", "band_high", "prior_same_style", "ambiguity_margin")})
strict = s.config.with_(band_low=0.85)                   # keep only HIGH-confidence pairs
s.run(parcels=[pid], config=strict)
print(s.parcel_summary().query("CADASTRE_POLYGON_PID == @pid")[["N_DUPLICATE_GROUPS", "N_REDUNDANT_ADDRESSES"]])
s.run(config=Config())                                   # back to defaults for everything

{'prior_same_style': -3.0, 'link_threshold': 0.5, 'ambiguity_margin': 1.5, 'band_high': 0.85, 'band_medium': 0.6, 'band_low': 0.4}


run: 1 parcels analysed in 0.10s, 299 duplicate groups
   N_DUPLICATE_GROUPS  N_REDUNDANT_ADDRESSES
0                 299                    449


run: 32 parcels analysed in 0.30s, 714 duplicate groups


AssociatorSession(2350 rows, 32 parcels analysed, 714 duplicate groups, 233 review pairs)

In [24]:
s

AssociatorSession(2350 rows, 32 parcels analysed, 714 duplicate groups, 233 review pairs)

## 11 · Residential use reconciliation

Two sources say whether an `ADDR_ID` is a residential dwelling:
- **base:** `ADDRESS_TYPE`, where `RESIDENTIAL_DWELLING` and `SHELL_CHILD` count as residential;
- **other:** the second dataset's `RESIDENTIAL_DWELLING` flag.

Neither is trusted outright. The decision is made per **premise**: a duplicate group (one physical place with several `ADDR_ID`s) or a lone address. It combines four kinds of evidence:
- the structural prior: flat type, and whether an address heads a building or site (not itself a dwelling) or stands alone;
- consensus among other premises in the same block and on the same floor;
- both sources' flags, weighted by how reliable each has proven in that context.

A residential premise is counted **once**, on one representative `ADDR_ID`.

**Areas:** when one source says residential and the other doesn't for a whole block, and the same scheme or building elsewhere on the parcel agrees, that's a **coverage gap** in the silent source. That source's flags are set aside for the block. Without that support, the block is flagged as a **block disagreement** for area review.

In [25]:
s.attach_use(use_df)

attach_use: 2350 flags (1123 residential), 1 issue types


,CHECK,SEVERITY,ROWS,DETAIL,EXAMPLE_ADDR_IDS
0,flags_loaded,info,2350,2350 flags loaded (1123 residential),


In [26]:
s.reconcile_use()
s.use_category_summary()

reconcile_use: 2350 addresses in 1420 premises; 1128 base-only, 19 other-only; 924 suggested changes to base, 74 to other


DIRECTION,CATEGORY,MEANING,BASE_ONLY,BOTH,NEITHER,OTHER_ONLY
0,BUILDING_OR_SITE_ADDRESS,"address of a building or site that holds units, or a shell found by detect_s...",14,5,0,0
1,DUPLICATE_OF_COUNTED_DWELLING,same dwelling as another ADDR_ID that is (or should be) the one counted,883,0,0,0
2,LIKELY_NOT_RESIDENTIAL,"flagged residential, but the premise looks non-residential",10,0,0,3
3,MISSED_BY_BASE,residential premise the base ADDRESS_TYPE does not count,0,0,0,12
4,MISSED_BY_OTHER,residential premise the second dataset does not count,66,0,0,0
5,OK,sources agree with the reconciled use,0,1099,99,0
6,POSSIBLE_DUPLICATE,may be the same dwelling as a review-candidate address; no change suggested ...,153,0,0,4
7,UNRESOLVED,sources disagree and the evidence is balanced,2,0,0,0


How far each source can be trusted, fitted from the data per context. Sensitivity is how often it flags real dwellings; specificity is how often it leaves non-dwellings unflagged.

In [27]:
s.source_reliability()

,SOURCE,CONTEXT,SENSITIVITY,SPECIFICITY,EST_DWELLINGS,EST_NON_DWELLINGS
0,BASE,ancillary,0.899,0.776,0.1,20.9
1,BASE,building,0.902,0.611,0.2,40.8
2,BASE,commercial,0.894,0.866,0.2,26.8
3,BASE,other_sub,0.899,0.909,0.0,1.0
4,BASE,standalone,0.971,0.815,24.0,1.0
5,BASE,unit,0.995,0.718,1212.4,92.6
6,OTHER,ancillary,0.892,0.936,0.1,20.9
7,OTHER,building,0.901,0.886,0.2,40.8
8,OTHER,commercial,0.894,0.921,0.2,26.8
9,OTHER,other_sub,0.899,0.909,0.0,1.0


### Areas to rectify (one row per block: a numbering scheme within one building)

In [28]:
s.use_areas()

,CADASTRE_POLYGON_PID,BLOCK_ID,STRUCTURE_ID,STYLE_TEMPLATE,CONTEXT,N_PREMISES,BOTH_RESIDENTIAL,BASE_ONLY,OTHER_ONLY,NEITHER,RECONCILED_RESIDENTIAL,VERDICT
0,cpl5a1700003dde,S1.3,B3,"UNIT {B}{L}{UU}, {N}-{N} {STREET}",unit,48,0,47,0,1,47,OTHER_COVERAGE_GAP
1,cpl5a1700003dde,S2.3,B3,"TOWER {B}, UNIT {L}{UU}, {N}-{N} {STREET}",unit,35,0,35,0,0,35,OTHER_COVERAGE_GAP
2,cpl5a1700003dde,S3.2,B3,"UNIT {L}{UU}, {N} {STREET}",unit,28,0,28,0,0,28,OTHER_COVERAGE_GAP


### Base says residential, the other dataset does not

Each such address gets a category explaining why, and a suggested fix.

In [29]:
focus = s.use_reconciliation(direction="BASE_ONLY")
focus.CATEGORY.value_counts()

CATEGORY
DUPLICATE_OF_COUNTED_DWELLING    883
POSSIBLE_DUPLICATE               153
MISSED_BY_OTHER                   66
BUILDING_OR_SITE_ADDRESS          14
LIKELY_NOT_RESIDENTIAL            10
UNRESOLVED                         2
Name: count, dtype: int64

In [30]:
cols = ["ADDR_ID", "LABEL", "BASE_ADDRESS_TYPE", "OTHER_RESIDENTIAL", "RECONCILED_USE", "P_DWELLING",
        "CATEGORY", "SUGGESTED_ACTION", "AREA_PATTERN"]
focus[focus.CATEGORY != "DUPLICATE_OF_COUNTED_DWELLING"][cols].head(15)

,ADDR_ID,LABEL,BASE_ADDRESS_TYPE,OTHER_RESIDENTIAL,RECONCILED_USE,P_DWELLING,CATEGORY,SUGGESTED_ACTION,AREA_PATTERN
0,A0001965,"SUITE 107, 64-68 BANKSIA ST",SHELL_CHILD,False,NOT_RESIDENTIAL,0.0134,LIKELY_NOT_RESIDENTIAL,base -> not residential,
1,A0001967,"SUITE 108, 64-68 BANKSIA ST",SHELL_CHILD,False,NOT_RESIDENTIAL,0.0134,LIKELY_NOT_RESIDENTIAL,base -> not residential,
3,A0001977,"UNIT 204, 64-68 BANKSIA ST",SHELL_CHILD,False,RESIDENTIAL,0.9406,MISSED_BY_OTHER,other -> residential,
52,A0002048,"UNIT 707, 64-68 BANKSIA ST",SHELL_CHILD,False,RESIDENTIAL,0.9406,MISSED_BY_OTHER,other -> residential,
213,A0000485,"UNIT A1209, 1-15 MERIDIAN AV",SHELL_CHILD,False,RESIDENTIAL,0.9406,MISSED_BY_OTHER,other -> residential,
273,A0000587,"UNIT A1701, 1-15 MERIDIAN AV",SHELL_CHILD,False,RESIDENTIAL,0.9406,MISSED_BY_OTHER,other -> residential,
298,A0000638,7 MERIDIAN AV,RESIDENTIAL_DWELLING,False,NOT_RESIDENTIAL,0.0006,BUILDING_OR_SITE_ADDRESS,base -> not residential,
322,A0000678,"UNIT B208, 1-15 MERIDIAN AV",SHELL_CHILD,False,RESIDENTIAL,0.9406,MISSED_BY_OTHER,other -> residential,
356,A0000733,"UNIT B506, 1-15 MERIDIAN AV",SHELL_CHILD,False,RESIDENTIAL,0.9406,MISSED_BY_OTHER,other -> residential,
421,A0000844,"UNIT B1104, 1-15 MERIDIAN AV",SHELL_CHILD,False,RESIDENTIAL,0.9406,MISSED_BY_OTHER,other -> residential,


Per parcel: each source's residential count against the reconciled dwelling count, worst first.

In [31]:
s.use_summary().head(10)

,CADASTRE_POLYGON_PID,N_ADDRESSES,BASE_RESIDENTIAL,OTHER_RESIDENTIAL,RECONCILED_DWELLINGS,UNCERTAIN_ADDRESSES,BASE_ONLY,OTHER_ONLY,DUPLICATE_OF_COUNTED_DWELLING,BUILDING_OR_SITE_ADDRESS,MISSED_BY_OTHER,MISSED_BY_BASE,MISSED_BY_BOTH,LIKELY_NOT_RESIDENTIAL,POSSIBLE_DUPLICATE,UNRESOLVED,BASE_MINUS_RECONCILED,OTHER_MINUS_RECONCILED
0,cpl5a1700003dde,790,762,271,323,1,497,6,442,2,52,6,0,1,0,0,439,-52
1,cpl5a170000f778,212,207,116,120,85,91,0,0,2,0,0,0,0,89,0,87,-4
2,cpl5a1700007bbc,118,114,62,63,53,54,2,0,0,0,0,0,0,56,0,51,-1
3,cpl5a170000b99a,159,155,59,59,0,97,1,97,0,0,1,0,0,0,0,96,0
4,cpl5a170000d889,210,206,116,120,0,91,1,87,0,4,1,0,0,0,0,86,-4
5,cpl5a1700001eef,201,190,110,112,0,82,2,78,0,3,1,0,2,0,0,78,-2
6,cpl5a1700005ccd,117,115,63,64,0,52,0,50,1,1,0,0,0,0,0,51,-1
7,cpl5a1700013556,168,138,87,88,0,53,2,49,0,2,1,0,3,0,0,50,-1
8,cpl5a1700009aab,91,89,52,54,0,38,1,36,0,2,1,0,0,0,0,35,-2
9,cpl5a1700015445,97,79,47,48,0,33,1,25,0,2,0,0,7,0,0,31,-1


In [32]:
aid = s.use_reconciliation(category="MISSED_BY_OTHER").ADDR_ID.iloc[0]
_ = s.explain(aid)

ADDR_ID          A0001977
SOURCE_ID        TXT40025701
PARCEL           cpl5a1700013556
LABEL            UNIT 204, 64-68 BANKSIA ST
KIND             sub-address
STYLE            S1  UNIT {L}{UU}, {N}-{N} {STREET}
BLOCK            S1.1  64-68 BANKSIA ST
NUMBERING_MODEL  unit number encodes level (1103 = level 11, unit 3); levels L2-L12, ~8 per level
KEYS             {'raw': '204', 'lu': (2.0, '4'), 'lr': (2.0, 4), 'seq': 4}
STRUCTURE        B1
PARENT_ROLE      
GROUP            G14 (HIGH, preferred A0001977)
USE              RESIDENTIAL (P=0.94) base=True other=False -> MISSED_BY_OTHER other -> residential
USE_EVIDENCE     prior (unit) +1.5; block peers +3.0; base +1.3; other -3.0


Scoring against the synthetic truth. A source is right for a physical place when it flags exactly one of its `ADDR_ID`s if the place is a dwelling, and none otherwise. The table compares before and after applying the suggested flags.

In [33]:
ev = s.evaluate_use(truth)
ev["by_source"]

use: decided accuracy 0.9964, uncertain 6.0%, 1193 true dwellings


,SOURCE,COUNT_BEFORE,COUNT_AFTER,ENTITIES_RIGHT_BEFORE,ENTITIES_RIGHT_AFTER,CHANGES_SUGGESTED,RIGHT_AFTER_BUILDING_FLAGGED,RIGHT_AFTER_COMMERCIAL_FLAGGED,RIGHT_AFTER_DWELLING_NOT_FLAGGED,RIGHT_AFTER_BLOCK_MISSED,RIGHT_AFTER_DWELLING_MISSED
0,BASE,2232,1332,0.3438,0.8818,924,16/17,8/8,31/32,NaN,NaN
1,OTHER,1123,1181,0.9405,0.9906,74,NaN,3/3,NaN,47/48,19/25


The worst parcel's use plan, saved as HTML. Cells are coloured by reconciled use, with a red outline where a use correction is suggested.

In [34]:
worst = s.use_summary().CADASTRE_POLYGON_PID.iloc[0]
_ = s.save_html("../outputs/use_%s.html" % worst, parcels=[worst])

saved ..\outputs\use_cpl5a1700003dde.html (1 parcels in detail, 398 KB)


## 12 · Shell addresses

A **shell** address stands for something other than one dwelling or tenancy:
- **BUILDING:** the whole building, e.g. `10 CORAL ST` over its units;
- **SITE:** several buildings or lots, e.g. a range address over two towers, or an association lot whose townhouses sit on their own parcels;
- **COMMON_PROPERTY:** `LOBBY`, `CARPARK 1`, `BASEMENT 1`, `SUBSTATION`, a `COMMON PROPERTY` building name;
- **FLOOR:** `LEVEL 3` with no flat. A whole floor is offered for review only.

Each address gets a log-odds score from:
- what kind of address it is;
- for parents (no flat or level), how many distinct sub-address premises hang below the same base, through the parcel's inferred structures and on neighbouring parcels;
- optionally, geometry, at two levels:
  - **parcel level (no geocode needed):** does one building hold all the parcel's premises? Are there more buildings than premises, or a few premises spread over several buildings (a front house)? Is there no building at all, with the premises next door (an association lot)?
  - **building level:** which footprint a geocode falls in, and whether a parent sits alone in its own footprint with its units elsewhere.

Most G-NAF geocodes sit at the **parcel centroid**. Those say nothing about which building an address is in, so they are marked `GEOCODE_LEVEL = PROPERTY` and only the parcel-level rules use them. The synthetic data mirrors this: 85% of its geocodes are parcel centroids.

Private car spaces and stores are someone's lot, so they are neither dwellings nor shells.

Geometry is optional. Without it, the same detection runs on text and parcel linkage alone.

In [35]:
s.attach_geometry(geo["cadastre"], geo["buildings"])

attach_geometry: 65 parcel polygons, 90 footprints (lon/lat)


,CHECK,SEVERITY,ROWS,DETAIL,EXAMPLE_IDS
0,parcels_loaded,info,65,65 parcel polygons loaded,
1,buildings_loaded,info,90,90 footprint-parcel links loaded,
2,coordinates,info,1,geometry read as longitude/latitude (projected per parcel to local metres),


In [36]:
s.detect_shells()
s.shell_category_summary()

detect_shells: 2350 addresses, 49 shells (building 38, common_property 5, site 6), 5 possible; 17 shells counted as residential; 2350 addresses placed on geometry
  (run reconcile_use() again so the use reconciliation uses these verdicts)


VERDICT,CATEGORY,MEANING,NOT_SHELL,POSSIBLE_SHELL,SHELL
0,OK,verdict and ADDRESS_TYPE agree (or there is nothing to compare),2296,0,32
1,REVIEW,possible shell: the evidence is not conclusive,0,5,0
2,SHELL_FLAGGED_RESIDENTIAL,shell address counted as a residential dwelling by ADDRESS_TYPE or the secon...,0,0,17


In [37]:
s.shells(include_not_shell=True).GEOCODE_LEVEL.value_counts()

GEOCODE_LEVEL
PROPERTY     2208
FOOTPRINT     138
OPEN            4
Name: count, dtype: int64

In [38]:
cols = ["ADDR_ID", "LABEL", "VERDICT", "SHELL_CLASS", "P_SHELL", "N_CHILD_PREMISES", "N_CROSS_PARCEL_PREMISES",
        "BASE_ADDRESS_TYPE", "OTHER_RESIDENTIAL", "CATEGORY"]
s.shells(verdict="SHELL")[cols].head(15)

,ADDR_ID,LABEL,VERDICT,SHELL_CLASS,P_SHELL,N_CHILD_PREMISES,N_CROSS_PARCEL_PREMISES,BASE_ADDRESS_TYPE,OTHER_RESIDENTIAL,CATEGORY
0,A0000001,2-8 HARBOURVIEW PDE,SHELL,BUILDING,0.9939,116.0,0.0,SHELL_PARENT,False,OK
1,A0000002,2 HARBOURVIEW PDE,SHELL,BUILDING,0.9939,116.0,0.0,SHELL_PARENT,False,OK
2,A0000003,"ALDER TOWER, 2-8 HARBOURVIEW PDE",SHELL,BUILDING,0.9939,116.0,0.0,SHELL_PARENT,False,OK
3,A0000203,"TOWER A, 1-15 MERIDIAN AV",SHELL,BUILDING,0.9837,182.0,0.0,SHELL_PARENT,False,OK
4,A0000638,7 MERIDIAN AV,SHELL,BUILDING,0.9837,97.0,0.0,RESIDENTIAL_DWELLING,False,SHELL_FLAGGED_RESIDENTIAL
5,A0000202,1-15 MERIDIAN AV,SHELL,SITE,0.9837,331.0,0.0,SHELL_PARENT,False,OK
6,A0000880,15 MERIDIAN AV,SHELL,BUILDING,0.9837,48.0,0.0,SHELL_PARENT,False,OK
7,A0000204,1 MERIDIAN AV,SHELL,BUILDING,0.9837,182.0,0.0,SHELL_PARENT,False,OK
8,A0000637,"TOWER B, 1-15 MERIDIAN AV",SHELL,BUILDING,0.9837,97.0,0.0,SHELL_PARENT,False,OK
9,A0000879,"TOWER C, 1-15 MERIDIAN AV",SHELL,BUILDING,0.9837,48.0,0.0,RESIDENTIAL_DWELLING,False,SHELL_FLAGGED_RESIDENTIAL


### Shells counted as residential dwellings

These are building, site or common-property addresses that `ADDRESS_TYPE` or the second dataset counts as a dwelling.

In [39]:
s.shells(category="SHELL_FLAGGED_RESIDENTIAL")[cols + ["SUGGESTED_ACTION"]].head(10)

,ADDR_ID,LABEL,VERDICT,SHELL_CLASS,P_SHELL,N_CHILD_PREMISES,N_CROSS_PARCEL_PREMISES,BASE_ADDRESS_TYPE,OTHER_RESIDENTIAL,CATEGORY,SUGGESTED_ACTION
0,A0000638,7 MERIDIAN AV,SHELL,BUILDING,0.9837,97.0,0.0,RESIDENTIAL_DWELLING,False,SHELL_FLAGGED_RESIDENTIAL,base ADDRESS_TYPE RESIDENTIAL_DWELLING -> a shell type
1,A0000879,"TOWER C, 1-15 MERIDIAN AV",SHELL,BUILDING,0.9837,48.0,0.0,RESIDENTIAL_DWELLING,False,SHELL_FLAGGED_RESIDENTIAL,base ADDRESS_TYPE RESIDENTIAL_DWELLING -> a shell type
2,A0000993,2 WATTLE LANE,SHELL,BUILDING,0.9866,64.0,0.0,RESIDENTIAL_DWELLING,False,SHELL_FLAGGED_RESIDENTIAL,base ADDRESS_TYPE RESIDENTIAL_DWELLING -> a shell type
3,A0001796,52 SKYLINE BVD,SHELL,BUILDING,0.9644,60.0,0.0,RESIDENTIAL_DWELLING,False,SHELL_FLAGGED_RESIDENTIAL,base ADDRESS_TYPE RESIDENTIAL_DWELLING -> a shell type
4,A0001689,50 SKYLINE BVD,SHELL,BUILDING,0.9644,60.0,0.0,RESIDENTIAL_DWELLING,False,SHELL_FLAGGED_RESIDENTIAL,base ADDRESS_TYPE RESIDENTIAL_DWELLING -> a shell type
5,A0001900,5 ORCHARD LANE,SHELL,SITE,0.9644,8.0,0.0,RESIDENTIAL_DWELLING,False,SHELL_FLAGGED_RESIDENTIAL,base ADDRESS_TYPE RESIDENTIAL_DWELLING -> a shell type
6,A0002188,"CARPARK 1, 10 CORAL ST",SHELL,COMMON_PROPERTY,0.9526,NaN,NaN,RESIDENTIAL_DWELLING,False,SHELL_FLAGGED_RESIDENTIAL,base ADDRESS_TYPE RESIDENTIAL_DWELLING -> a shell type
7,A0002187,"LOBBY, 10 CORAL ST",SHELL,COMMON_PROPERTY,0.9526,NaN,NaN,RESIDENTIAL_DWELLING,False,SHELL_FLAGGED_RESIDENTIAL,base ADDRESS_TYPE RESIDENTIAL_DWELLING -> a shell type
8,A0002209,1 WARATAH CL,SHELL,SITE,0.9759,0.0,10.0,RESIDENTIAL_DWELLING,False,SHELL_FLAGGED_RESIDENTIAL,base ADDRESS_TYPE RESIDENTIAL_DWELLING -> a shell type
9,A0002228,1 SALTBUSH TCE,SHELL,BUILDING,0.9370,3.0,0.0,RESIDENTIAL_DWELLING,False,SHELL_FLAGGED_RESIDENTIAL,base ADDRESS_TYPE RESIDENTIAL_DWELLING -> a shell type


Per parcel, and per building footprint:

In [40]:
s.shell_summary().head(8)

,CADASTRE_POLYGON_PID,N_ADDRESSES,N_SHELLS,N_POSSIBLE,N_SHELLS_FLAGGED_RESIDENTIAL,N_GEOCODED,N_BUILDING_LEVEL_GEOCODES,BUILDING,SITE,COMMON_PROPERTY,FLOOR,N_FOOTPRINTS,PARCEL_AREA_M2
0,cpl5a1700003dde,790,7,0,2,790,103,6,1,0,0,3,9800.0
1,cpl5a1700015445,97,6,0,2,97,1,1,0,5,0,2,2600.0
2,cpl5a170000f778,212,5,0,2,212,1,4,1,0,0,2,4500.0
3,cpl5a170000d889,210,3,0,0,210,24,2,1,0,0,2,4500.0
4,cpl5a1700001eef,201,3,0,0,201,0,3,0,0,0,1,2300.0
5,cpl5a1700013556,168,2,0,0,168,0,2,0,0,0,1,2116.0
6,cpl5a170000b99a,159,2,0,0,159,0,2,0,0,0,1,1840.0
7,cpl5a1700005ccd,117,2,0,1,117,0,2,0,0,0,1,1600.0


In [41]:
cp = s.shell_summary().sort_values("COMMON_PROPERTY", ascending=False).CADASTRE_POLYGON_PID.iloc[0]
s.shell_buildings(cp)

,CADASTRE_POLYGON_PID,BUILDING_ID,AREA_M2,N_BUILDING_LEVEL_GEOCODES,N_SUB_PREMISES,N_PARENTS,N_SHELLS,SHELLS
0,cpl5a1700015445,BLD000023,1088.0,0,0,0,0,
1,cpl5a1700015445,BLD000024,12.0,0,0,0,0,


Shell verdicts also feed the residential-use reconciliation:
- a shell is treated as a building or site address, never a dwelling;
- a parent the detector clears is treated as a standalone dwelling;
- a possible shell (such as a front house) gets a mild prior, so the two sources decide.

In [42]:
s.reconcile_use()
s.evaluate_use(truth)["by_source"]

reconcile_use: 2350 addresses in 1420 premises; 1128 base-only, 19 other-only; 920 suggested changes to base, 69 to other
use: decided accuracy 0.9986, uncertain 6.0%, 1193 true dwellings


,SOURCE,COUNT_BEFORE,COUNT_AFTER,ENTITIES_RIGHT_BEFORE,ENTITIES_RIGHT_AFTER,CHANGES_SUGGESTED,RIGHT_AFTER_BUILDING_FLAGGED,RIGHT_AFTER_COMMERCIAL_FLAGGED,RIGHT_AFTER_DWELLING_NOT_FLAGGED,RIGHT_AFTER_BLOCK_MISSED,RIGHT_AFTER_DWELLING_MISSED
0,BASE,2232,1336,0.3438,0.8865,920,17/17,8/8,31/32,NaN,NaN
1,OTHER,1123,1186,0.9405,0.9945,69,NaN,3/3,NaN,47/48,19/25


A front house with units behind it. Its base address has sub-addresses, so text alone would call it a building shell. The parcel's footprints say otherwise: three units spread over two buildings. With only a centroid geocode the evidence goes no further than review (`POSSIBLE_SHELL`). A building-level geocode in the house's own footprint would clear it outright.

In [43]:
allrows = s.shells(include_not_shell=True)
fh = allrows[allrows.EVIDENCE.str.contains("front house")].sort_values("P_SHELL").ADDR_ID.iloc[0]
_ = s.explain(fh)

ADDR_ID          A0002318
SOURCE_ID        TXT40030134
PARCEL           cpl5a170006866a
LABEL            19 SALTBUSH TCE
KIND             parent
STYLE            
BLOCK            
NUMBERING_MODEL  
KEYS             {}
STRUCTURE        B1
PARENT_ROLE      BUILDING
GROUP            not grouped
USE              RESIDENTIAL (P=0.96) base=True other=True -> OK
USE_EVIDENCE     prior (possible_shell) -1.0; base +2.1; other +2.1
SHELL            POSSIBLE_SHELL BUILDING (P=0.51) REVIEW review
SHELL_EVIDENCE   parent -1.5; 2 sub-address premises below it +2.5; 2 premises spread over 2 buildings: the base address may be a front house -1.0


Scoring against the synthetic truth. `possible` counts the `POSSIBLE_SHELL` verdicts left for review.

In [44]:
ev = s.evaluate_shells(truth)
ev["by_scenario"]

shells: precision 1.0000  recall 1.0000  (TP 49, FP 0, FN 0); class accuracy 1.0; possible: 0 true / 5 not


,SCENARIO,PRECISION,RECALL,F1,TP,FP,FN,CLASS_ACCURACY,POSSIBLE_TRUE,POSSIBLE_FALSE
0,background,1.0,1.0,1.0,14,0,0,1.0,0,5
1,common_property,1.0,1.0,1.0,6,0,0,1.0,0,0
2,community_title,1.0,1.0,1.0,1,0,0,1.0,0,0
3,corner_frontage,1.0,1.0,1.0,2,0,0,1.0,0,0
4,level_convention,1.0,1.0,1.0,1,0,0,1.0,0,0
5,mixed_use,1.0,1.0,1.0,2,0,0,1.0,0,0
6,multi_tower,1.0,1.0,1.0,7,0,0,1.0,0,0
7,single_tower,1.0,1.0,1.0,3,0,0,1.0,0,0
8,staggered_coverage,1.0,1.0,1.0,2,0,0,1.0,0,0
9,strata_lot_numbering,1.0,1.0,1.0,1,0,0,1.0,0,0


In [45]:
ev["confusion"]

PREDICTED,POSSIBLE_SHELL BUILDING,SHELL BUILDING,SHELL COMMON_PROPERTY,SHELL SITE
TRUE_CLASS,,,,
(none),5,0,0,0
BUILDING,0,38,0,0
COMMON_PROPERTY,0,0,5,0
SITE,0,0,0,6


The parcel with the most common property, saved as HTML. Section 8 draws the parcel (dashed), its footprints and geocodes, coloured by verdict. Hollow dots are property-level geocodes.

In [46]:
_ = s.save_html("../outputs/shells_%s.html" % cp, parcels=[cp])

saved ..\outputs\shells_cpl5a1700015445.html (1 parcels in detail, 67 KB)


## 13 · Hand the results on

`to_frames()` returns every output as a DataFrame. Write them wherever your workflow needs, e.g. `frames["duplicate_groups"].to_parquet(...)` or `spark.createDataFrame(...)`.

In [47]:
frames = s.to_frames()
{name: f.shape for name, f in frames.items()}

{'issues': (1, 5),
 'parcel_summary': (32, 12),
 'styles': (66, 8),
 'blocks': (80, 14),
 'mappings': (84, 15),
 'structures': (43, 9),
 'addresses': (2317, 18),
 'duplicate_pairs': (1152, 17),
 'duplicate_groups': (714, 13),
 'review_candidates': (233, 17),
 'use_issues': (1, 5),
 'use_reconciliation': (2350, 24),
 'use_areas': (80, 12),
 'use_by_parcel': (65, 18),
 'source_reliability': (12, 6),
 'geometry_issues': (3, 5),
 'shells': (2350, 22),
 'shell_buildings': (90, 8),
 'shells_by_parcel': (65, 13),
 'shell_geocode_issues': (1, 5)}

## 14 · Save the full HTML report

One standalone page: headline numbers, an index of every analysed parcel, and a full schematic section for the `max_parcels` parcels with the most duplicate groups (default 50). Pass `parcels=[...]` to choose them yourself. Once `reconcile_use()` and `detect_shells()` have run, each section also includes the residential-use and shell parts.

In [48]:
_ = s.save_html("../outputs/parcel_address_report.html")

saved ..\outputs\parcel_address_report.html (32 parcels in detail, 1417 KB)
